In [1]:
import random
import numpy as np
import polars as pl
import torch
from sentence_transformers import SentenceTransformer, InputExample
from torch.utils.data import Sampler, DataLoader
from sentence_transformers.sentence_transformer import losses, evaluation
from torch.utils.data import DataLoader
import os
import re
import shutil

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME = "cointegrated/rubert-tiny2"
MAX_LEN = 128

In [3]:
train_df = pl.read_parquet(f"/kaggle/input/datasets/hxllmvdx/train-avito-ds-bootcamp/train.parquet")
items_df = pl.read_parquet(f"/kaggle/input/datasets/hxllmvdx/train-avito-ds-bootcamp/benchmark_items.parquet")

indexed_ids = set(items_df["item_id"].to_list())
train_filtered = train_df.filter(pl.col("item_id").is_in(indexed_ids))

GROUP_KEYS = ["search_query", "search_infm_params_text", "search_location_id",
              "search_category", "search_is_delivery_search"]

unique_queries = (
    train_filtered.group_by(GROUP_KEYS)
    .agg(pl.col("item_id").alias("relevant_items"))
    .sort(GROUP_KEYS)
)
eval_queries = unique_queries.sample(n=min(2000, len(unique_queries)), seed=SEED)

eval_texts = set(
    eval_queries["search_query"].str.to_lowercase().str.strip_chars().to_list()
)
train_ft = train_filtered.filter(
    ~pl.col("search_query").str.to_lowercase().str.strip_chars().is_in(eval_texts)
)
print(f"train rows: {len(train_filtered)} -> {len(train_ft)} после исключения eval")

train rows: 33010 -> 18108 после исключения eval


In [4]:
# группируем позитивы по query-ключу
pairs = (
    train_ft.group_by(GROUP_KEYS)
    .agg(pl.col("item_id").alias("pos_items"))
)

def make_examples(df, one_pos_per_query=True):
    examples = []
    for row in df.iter_rows(named=True):
        q = query_text(row["search_query"], row["search_infm_params_text"])
        pos = row["pos_items"]
        if one_pos_per_query:
            pos = [random.choice(pos)]
        for iid in pos:
            if iid in item_texts:
                examples.append(InputExample(texts=[q, item_texts[iid]]))
    return examples

In [5]:
RAW_KEYS = ["Сортировка для URL","Слова в описании","Вид услуги","Тип услуги","Тип товара",
    "Вид товара","Срочная услуга (мультистатус)","Кто оказывает услуги","Рейтинг пользователя",
    "Тип объявления","Сфера деятельности","График работы, дни недели","Марка","Состояние",
    "Вид техники","Открытие в сегменте авито для бизнеса","Марка авто","График работы v2",
    "График работы","Где вы оказываете услуги","Ваши клиенты","Специальность или сфера",
    "Онлайн-запись","Тип автосервиса","Тип услуги автосервиса","Работа по договору",
    "Гарантия","Предмет или специальность"]
SPECIAL_KEYS = {"Сортировка для URL", "Рейтинг пользователя", "Слова в описании"}

KEY_REGEX = re.compile(r"\b(" + "|".join(re.escape(k) for k in sorted(set(RAW_KEYS), key=len, reverse=True)) + ")")

def _is_boundary(word):
    if not word or word[0] in "[{": return True
    if not word[0].isupper(): return False
    letters = [c for c in word if c.isalpha()]
    if letters and all(c.isupper() for c in letters) and len(letters) <= 4: return False
    return True

def parse_params(text):
    if not text: return {}
    matches = list(KEY_REGEX.finditer(text))
    result = {}
    for i, m in enumerate(matches):
        seg = text[m.end(): matches[i+1].start() if i+1 < len(matches) else len(text)]
        tokens = [(t.start(), t.end(), t.group()) for t in re.finditer(r"[^\s]+", seg)]
        if not tokens: continue
        end = len(seg)
        for j in range(1, len(tokens)):
            if _is_boundary(tokens[j][2]):
                end = tokens[j][0]; break
        value = seg[tokens[0][0]:end].strip(" ,;.")
        if value: result.setdefault(m.group(1), []).append(value)
    return result

def item_text(title, desc, params_text):
    d = parse_params(params_text or "")
    flat = " | ".join(f"{k}: {' '.join(v)}" for k, v in d.items() if k not in SPECIAL_KEYS and v)
    return f"{title or ''} {flat} {(desc or '')}".strip()

def query_text(q, params_text):
    return f"{q or ''} {params_text or ''}".strip()

# маппинг item_id -> текст для эмбеддинга
item_texts = dict(zip(
    items_df["item_id"].to_list(),
    [item_text(t, d, p) for t, d, p in zip(
        items_df["item_title_raw"].to_list(),
        items_df["item_description_raw"].to_list(),
        items_df["item_infm_params_text"].to_list())]
))

In [6]:
@torch.no_grad()
def dense_recall(model, eval_df, item_ids, item_embs, k=50, chunk=512):
    q_texts = [query_text(r["search_query"], r["search_infm_params_text"])
               for r in eval_df.iter_rows(named=True)]
    q_embs = model.encode(q_texts, batch_size=512, normalize_embeddings=True,
                          convert_to_numpy=True, show_progress_bar=True)
    id_arr = np.array(item_ids)
    recalls = []
    rel_lists = eval_df["relevant_items"].to_list()
    for start in range(0, len(q_embs), chunk):
        sims = q_embs[start:start+chunk] @ item_embs.T
        top_idx = np.argpartition(-sims, k, axis=1)[:, :k]
        for j, rel in enumerate(rel_lists[start:start+chunk]):
            pred = set(id_arr[top_idx[j]].tolist())
            recalls.append(len(pred & set(rel)) / len(rel))
        del sims
    return float(np.mean(recalls))

In [7]:
model = SentenceTransformer(MODEL_NAME, device="cuda")
model.max_seq_length = MAX_LEN

mnrl = losses.MultipleNegativesRankingLoss(model)

item_ids = items_df["item_id"].to_list()

# --- Baseline: Recall@50 ДО обучения ---
item_embs = model.encode([item_texts[i] for i in item_ids], batch_size=512,
                         normalize_embeddings=True, convert_to_numpy=True,
                         show_progress_bar=True)
r0 = dense_recall(model, eval_queries, item_ids, item_embs)
print(f"baseline (no FT): dense Recall@50 = {r0:.4f}")

# расписание lr вместо пересоздания оптимизатора с одинаковым lr
LR_SCHEDULE = [2e-5, 2e-5, 1e-5, 1e-5, 5e-6, 5e-6]

history = []  # (tag, recall, path)

for epoch, lr in enumerate(LR_SCHEDULE):
    examples = make_examples(pairs, one_pos_per_query=True)
    loader = DataLoader(examples, shuffle=True, batch_size=256,
                        drop_last=True, num_workers=2)
    path = f"/content/rubert_ft_r1_ep{epoch}"
    model.fit(
        train_objectives=[(loader, mnrl)],
        epochs=1,
        warmup_steps=int(0.1 * len(loader)) if epoch == 0 else 0,
        optimizer_params={"lr": lr},
        use_amp=True,
        output_path=path,
        show_progress_bar=True,
    )
    # пересчитываем эмбеддинги корпуса ТЕКУЩЕЙ моделью
    item_embs = model.encode([item_texts[i] for i in item_ids], batch_size=512,
                             normalize_embeddings=True, convert_to_numpy=True,
                             show_progress_bar=True)
    r = dense_recall(model, eval_queries, item_ids, item_embs)
    history.append((f"r1_ep{epoch}", r, path))
    print(f"epoch {epoch} (lr={lr}): dense Recall@50 = {r:.4f} (baseline {r0:.4f})")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

...

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/370 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

epoch 5 (lr=5e-06): dense Recall@50 = 0.2329 (baseline 0.0421)


In [8]:
K_MINE = 100
NEG_PER_POS = 4  # 2 самых злых + 2 случайных из top-50

def norm(v):
    if v is None:
        return ""
    s = str(v).strip()
    return "" if s in ("0", "", "None") else s

cat_arr = np.array([norm(v) for v in items_df["item_category_id"].to_list()])
loc_arr = np.array([norm(v) for v in items_df["item_location_id"].to_list()])

q_rows = list(pairs.iter_rows(named=True))
q_texts = [query_text(r["search_query"], r["search_infm_params_text"]) for r in q_rows]
id_arr = np.array(item_ids)

hard_examples = []
chunk = 512
for start in range(0, len(q_rows), chunk):
    q_embs_chunk = model.encode(q_texts[start:start+chunk], batch_size=512,
                                normalize_embeddings=True, convert_to_numpy=True,
                                show_progress_bar=False)
    sims = q_embs_chunk @ item_embs.T  # item_embs — свежие, после раунда 1
    del q_embs_chunk

    for j, row in enumerate(q_rows[start:start+chunk]):
        positives = set(row["pos_items"])
        pos_id = random.choice(row["pos_items"])
        if pos_id not in item_texts:
            continue

        s = sims[j].copy()
        # фильтр как в проде: та же категория; для non-delivery — та же локация
        row_cat = str(row["search_category"])
        if row_cat not in ("0", "", "None"):
            s[cat_arr != row_cat] = -1e9
        is_delivery = bool(row["search_is_delivery_search"] or 0)
        row_loc = str(row["search_location_id"])
        if not is_delivery and row_loc not in ("0", "", "None"):
            s[loc_arr != row_loc] = -1e9

        top_idx = np.argpartition(-s, K_MINE)[:K_MINE]
        top_idx = top_idx[np.argsort(-s[top_idx])]  # по убыванию сходства
        candidates = [id_arr[t] for t in top_idx
                      if id_arr[t] not in positives and s[t] > -1e8]
        if not candidates:
            continue

        hardest = candidates[:2]                       # ранги 1-2
        rest = random.sample(candidates[2:50],         # + 2 случайных из top-50
                             min(2, len(candidates[2:50])))
        q = q_texts[start + j]
        for neg in (hardest + rest)[:NEG_PER_POS]:
            hard_examples.append(InputExample(texts=[q, item_texts[pos_id], item_texts[neg]]))
    del sims
    print(f"mined {min(start + chunk, len(q_rows))}/{len(q_rows)}", end="\r")

print(f"{len(hard_examples)} hard-negative примеров")

55502 hard-negative примеров


In [9]:
# микс: hard-примеры + 40% обычных пар (защита от катастрофического забывания)
mix_examples = hard_examples + random.sample(
    make_examples(pairs, one_pos_per_query=True),
    k=int(0.2 * len(hard_examples)),
)
random.shuffle(mix_examples)
print(f"раунд 2: {len(mix_examples)} примеров ({len(hard_examples)} hard + микс)")

hard_loader = DataLoader(mix_examples, shuffle=True, batch_size=128,
                         drop_last=True, num_workers=2)
mnrl2 = losses.MultipleNegativesRankingLoss(model)

path_r2 = "/kaggle/working/rubert_ft_r2"
model.fit(
    train_objectives=[(hard_loader, mnrl2)],
    epochs=1,
    warmup_steps=int(0.1 * len(hard_loader)),
    optimizer_params={"lr": 1e-5},
    use_amp=True,
    output_path=path_r2,
    show_progress_bar=True,
)

# ЧЕСТНЫЙ замер: пересчитываем эмбеддинги корпуса НОВОЙ моделью
item_embs = model.encode([item_texts[i] for i in item_ids], batch_size=512,
                         normalize_embeddings=True, convert_to_numpy=True,
                         show_progress_bar=True)
r2 = dense_recall(model, eval_queries, item_ids, item_embs)
history.append(("r2", r2, path_r2))
print(f"после раунда 2: dense Recall@50 = {r2:.4f}")

раунд 2: 66602 примеров (55502 hard + микс)


Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/370 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

после раунда 2: dense Recall@50 = 0.2444


In [12]:
history.sort(key=lambda x: -x[1])
for tag, r, p in history:
    print(f"{tag:>10}: {r:.4f}  ({p})")

best_tag, best_r, best_path = history[0]
print(f"\nЛучший чекпоинт: {best_tag} с Recall@50 = {best_r:.4f}")

FINAL = "/kaggle/working/rubert_ft_final"
if best_path != FINAL:
    if os.path.exists(FINAL):
        shutil.rmtree(FINAL)
    shutil.copytree(best_path, FINAL)

# эмбеддинги корпуса финальной (лучшей) моделью
best_model = SentenceTransformer(FINAL, device="cuda")
best_model.max_seq_length = MAX_LEN
final_embs = best_model.encode([item_texts[i] for i in item_ids], batch_size=512,
                               normalize_embeddings=True, convert_to_numpy=True,
                               show_progress_bar=True)
np.save("/kaggle/working/item_embeddings.npy", final_embs)
shutil.make_archive("finetuned_rubert", "zip", FINAL)
print("сохранено: rubert_ft_final.zip + item_embeddings.npy")

        r2: 0.2444  (/kaggle/working/rubert_ft_r2)
    r1_ep4: 0.2340  (/content/rubert_ft_r1_ep4)
    r1_ep3: 0.2337  (/content/rubert_ft_r1_ep3)
    r1_ep5: 0.2329  (/content/rubert_ft_r1_ep5)
    r1_ep2: 0.2314  (/content/rubert_ft_r1_ep2)
    r1_ep1: 0.2270  (/content/rubert_ft_r1_ep1)
    r1_ep0: 0.1760  (/content/rubert_ft_r1_ep0)

Лучший чекпоинт: r2 с Recall@50 = 0.2444


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Batches:   0%|          | 0/370 [00:00<?, ?it/s]

сохранено: rubert_ft_final.zip + item_embeddings.npy
